# Image evaluation: a classifier we can replace

**This step:** obtain image measurements and inspect them before implementing automated selection.

ImageNet is the training dataset/category set. Our first evaluator is a frozen **ResNet-18 trained on ImageNet-1K**, with one softmax score for each of 1,000 classes. A classifier answers a different question from a selection algorithm: the latter decides what to retain and reproduce.

Run from the project root with:

```sh
uv sync --locked --extra imagenet
uv run --extra imagenet jupyter lab notebooks/02_image_evaluation.ipynb
```

The first classifier load downloads about 45 MB from PyTorch into `.cache/imagenet`; subsequent runs use the cached file. No training occurs. This notebook is independent of notebook 01.

In [ ]:
from pathlib import Path
from html import escape
import random
import time

import numpy as np
from PIL import Image
from IPython.display import display
import ipywidgets as widgets

from automated_picbreeder.cppn import load_config, new_genome, render, png_bytes
from automated_picbreeder.evaluation import Evaluation, ImageEvaluator, validate_image
from automated_picbreeder.imagenet import ImageNetEvaluator

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## 1. Render a fixed set of candidates

Use nine initial candidates with seed 7 (set the human playground to seed 7 to match). The CPPN produces HSB values, which the shared renderer converts to uint8 RGB arrays. Keeping the candidates fixed lets us compare evaluators on the same inputs.

Later, images could come from selected or saved genomes. The evaluator only sees images; it has no dependency on CPPN or NEAT code.

In [ ]:
config = load_config()
rng = random.Random(7)
genomes = [new_genome(i, config, rng) for i in range(9)]
images = [render(genome, config, size=96) for genome in genomes]
print(f"{len(images)} images; shape {images[0].shape}; dtype {images[0].dtype}")

## 2. Load the frozen classifier

`ImageEvaluator` describes just one method: `evaluate(images) -> Evaluation`. An evaluation contains an image-by-measurement array, column names and provenance metadata. It does not choose any candidates.

ResNet-18 is a manageable first model. We explicitly choose `IMAGENET1K_V1`, rather than a moving `DEFAULT` alias, and use its official preprocessing. CPU is the default. Other supported Torchvision ImageNet classifiers can be chosen via `model_name` and a concrete `weights` version; a completely different evaluator need not use Torchvision at all.

In [ ]:
classifier = ImageNetEvaluator(
    model_name="resnet18",
    weights="IMAGENET1K_V1",
    device="cpu",
    batch_size=16,
    cache_dir=ROOT / ".cache" / "imagenet",
)
evaluator: ImageEvaluator = classifier
classifier.describe()

## 3. Look at the actual classifier input

The network receives RGB images converted from our CPPN’s HSB outputs. Legacy grayscale inputs are also supported by repeating brightness across three channels. The official transform resizes the shorter side to 256 pixels, takes a **central 224 × 224 crop**, rescales to [0, 1], and applies fixed channel normalization.

This removes part of the border. It matters for CPPNs: the classifier does not assess the entire original image. The preview below reverses only channel normalization so you can see the crop. The returned tensor is the normalized input actually used by the model.

In [ ]:
tensor = classifier.preprocess(images[2])
preprocessing = classifier.describe()["preprocessing"]
mean = np.array(preprocessing["mean"])
std = np.array(preprocessing["std"])
crop = tensor.numpy().transpose(1, 2, 0) * std + mean
crop = np.rint(np.clip(crop, 0, 1) * 255).astype(np.uint8)
display(widgets.HBox([
    widgets.VBox([widgets.Label("Original"), widgets.Image(value=png_bytes(images[2]), width=224, height=224)]),
    widgets.VBox([widgets.Label("Classifier crop"), widgets.Image(value=png_bytes(crop), width=224, height=224)]),
]))

## 4. Score every image against every class

`values[i, j]` is image i's score for class index j. Names are display labels; some labels repeat, so use the **column index** to identify a class.

We show just the three largest scores per image, but keep the full 9 × 1,000 matrix for the eventual archive. These are relative softmax scores, not a calibrated probability that an abstract pattern depicts a real object, and not a measure of naturalness or interestingness.

In [ ]:
started = time.perf_counter()
result = evaluator.evaluate(images)
print(f"Scored {len(images)} images in {time.perf_counter() - started:.2f}s")
print("Full score matrix:", result.values.shape)
print("Row sums:", result.values.sum(axis=1))


def show_scores(images, evaluation):
    """Display measurements without making a selection decision."""
    cards = []
    for i, (image, values) in enumerate(zip(images, evaluation.values)):
        top = np.argsort(-values, kind="stable")[:3]
        rows = "<br>".join(
            f"{escape(evaluation.names[j])} [{j}]: {values[j]:.4f}" for j in top
        )
        cards.append(widgets.VBox([
            widgets.Label(f"Candidate {i}"),
            widgets.Image(value=png_bytes(image), width=150, height=150),
            widgets.HTML(rows),
        ], layout=widgets.Layout(padding="8px")))
    display(widgets.GridBox(cards, layout=widgets.Layout(
        grid_template_columns="repeat(3, minmax(180px, 1fr))",
        grid_gap="8px", width="100%", max_width="850px",
    )))


show_scores(images, result)

## 5. Replace the evaluator

Here is a complete alternative with one column: pixel contrast (standard deviation after division by 255). It implements the same interface without inheriting a base class or knowing about ImageNet.

**Contrast is only an interface demonstration.** It is not a validated measure of interestingness, and optimising it can reward unhelpful patterns. Notice that measurements need not sum to 1 or even be probabilities. A selection policy must decide how to use them.

In [ ]:
class ContrastEvaluator:
    def evaluate(self, images):
        for image in images:
            validate_image(image)
        values = np.array([image.astype(float).std() / 255 for image in images]).reshape(-1, 1)
        return Evaluation(
            values=values,
            names=("pixel contrast",),
            metadata={"evaluator": "pixel_contrast", "definition": "std of all uint8 channel values / 255"},
        )


alternative: ImageEvaluator = ContrastEvaluator()
contrast_result = alternative.evaluate(images)
show_scores(images, contrast_result)

## 6. Compare without running an evolutionary search

For inspection, choose a single class index and compare the candidate it scores highest with the highest-contrast candidate. This is a small comparison of measurements, not the MAP-Elites rule.

Change `class_index` to any value in [0, 999]. In the later classifier archive, every class will retain its own champion; an offspring can improve several classes at once. We will not collapse all classes to each image's top confidence.

In [ ]:
class_index = int(result.values[0].argmax())  # Change to any ImageNet class index.
classifier_choice = int(result.values[:, class_index].argmax())
contrast_choice = int(contrast_result.values[:, 0].argmax())
print(f"Highest score for {result.names[class_index]} [{class_index}]: candidate {classifier_choice}")
print(f"Highest contrast: candidate {contrast_choice}")
display(widgets.HBox([
    widgets.VBox([widgets.Label("Chosen class"), widgets.Image(value=png_bytes(images[classifier_choice]), width=192, height=192)]),
    widgets.VBox([widgets.Label("Contrast"), widgets.Image(value=png_bytes(images[contrast_choice]), width=192, height=192)]),
]))

## What remains separate

- **Representation and variation:** CPPN rendering and NEAT mutation produce candidates.
- **Evaluation:** an interchangeable object returns measurements. History-dependent novelty can use an explicit reference set and record that set's identity in metadata; updating the set belongs to the search procedure.
- **Selection/search:** later code chooses parents and decides what to retain. MAP-Elites, a novelty archive, tournament selection and human choice can use different rules. Quality-diversity methods may interpret some measurements as descriptors and others as quality; the policy must specify that mapping.

The next implementation is the classifier-guided archive and its update rule. This notebook does not implement the archive, update novelty history or run an evolutionary search. A human ranking of nine candidates is also distinct from uniformly sampling class champions from an archive.

For now, compare classifier scores with your own perception of the images. Does a strong score indicate recognisable structure, or just a pattern that activates that class?

References: [Torchvision ResNet-18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html), [model/checkpoint API](https://docs.pytorch.org/vision/stable/models.html), and [the experiment brief](../docs/experiment-brief.md).